[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# A Frost Forecast


## What you will be able to do

Build a frost forecast from start to finish in LightGBM and again in XGBoost: choose each model's
settings on the validation stations, measure the chosen model once on the test stations, read what it
relies on, and save it to forecast a night it has never seen.

## The idea

### The problem

A road authority grits the roads at stations where frost is likely, and wants each evening's chance
that the day's lowest temperature falls below 0 degrees. Gritting too little leaves ice, and gritting
too much wastes salt, so a bare yes or no is less use than a probability that can be weighed against
the cost of each.

Earlier notebooks in this guide taught the parts of the answer: the classifiers and their
probabilities, early stopping, choosing settings without the test stations, reading a model's
contributions, saving a booster. Put together in the wrong order, the parts give a forecast that looks
better than it is, because the test stations helped choose it, or because the answer leaked into the
features. Here they go together in an order that keeps each kind of data to its one job.

### A formal definition

> A **probability forecast** gives each case a probability of the event, here frost. Its **log loss**
> is the average, over the cases, of minus the logarithm of the probability it gave to what actually
> happened, so a confident miss costs far more than a hesitant one. A **threshold** turns a
> probability into a decision, frost or not, after the probabilities have been judged.

### Why it works that way

The training stations fit each candidate model, the validation stations choose among the candidates
and decide when each stops adding trees, and the test stations measure the chosen model once. The
validation stations are used twice, for stopping and for choosing, which can make the chosen model's
validation score optimistic. The test score has no such bias, because nothing was decided
with it.

The candidates are compared by log loss, because accuracy at a threshold of one half throws the
probabilities away. With frost on about half the days, always forecasting frost is right about half
the time, which every model has to beat by a wide margin. Two models with the same accuracy can still
differ in how sure they are, and that difference is what a road authority weighing salt against ice
needs to know.

### Where this shows up

A road authority's gritting plan, a farm's frost alarm and an airport's de-icing roster all start from
a probability like this one. In this guide, the **Classification** notebook covered the classifiers, their
probabilities and their metrics, the **Early Stopping** notebook the number of trees, the **Tuning**
notebook settings chosen without the test stations, the **Reading What They Learned** notebook the
contributions, and the **Saving and Loading Boosters** notebook the files.

### What this notebook covers

- the frost labels, and a baseline that always forecasts frost
- settings for each library chosen on the validation stations by log loss
- the chosen probabilities checked against how often frost happened
- the test stations measured once, and what the chosen model relies on
- both models saved, loaded and asked about a planned night
- three mistakes a whole project can make

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
from sklearn.metrics import log_loss

frost = [1, 1, 0, 0]
print(round(log_loss(frost, [0.9, 0.8, 0.2, 0.1]), 3))
print(round(log_loss(frost, [0.6, 0.6, 0.4, 0.4]), 3))
print(round(log_loss(frost, [0.99, 0.99, 0.01, 0.99]), 3))
```

```
0.164
0.511
1.159
```

The first two forecasts are both right about every day at a threshold of one half, and the first, the
surer of the two, has the lower log loss. The third is surer still and wrong about one day, and that
one confident miss gives it the highest log loss of the three.

## Setup

The standard library's tools for files, the two boosting libraries, the table, and three ways to score
a forecast.

- `Path` names the saved files and `shutil` removes their folder at the end
- `numpy` and `pandas` build the table and the tables of results
- `lightgbm` and `xgboost` are LightGBM and XGBoost, and the cell prints their versions
- `accuracy_score`, `log_loss` and `roc_auc_score` score a forecast by its decisions, by its
  probabilities, and by how well it ranks frost days above the rest

`station_days` makes the same 20,000 made-up station-days as the rest of the guide, from a fixed
seed, so the table is the same on every run, and the cell splits the stations into 300 for training,
50 for validation and 50 for testing. The saved models go in `scratch`, which the cell makes and the
notebook's last cell removes.

In [1]:
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

scratch = Path("scratch")
scratch.mkdir(exist_ok=True)
print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


## Worked examples

### The frost labels and a baseline

A day is a frost day when its lowest temperature is below 0. The baseline forecasts frost every day,
with a probability equal to the share of frost days in training.

In [2]:
frost_train, frost_valid, frost_test = [(rows["tmin"] < 0).astype(int) for rows in (train, valid, test)]
print("share of frost days:", round(frost_train.mean(), 3), round(frost_valid.mean(), 3), round(frost_test.mean(), 3))

baseline = np.full(len(frost_valid), frost_train.mean())
print("baseline on the validation stations | accuracy", round(accuracy_score(frost_valid, baseline > 0.5), 3),
      "| log loss", round(log_loss(frost_valid, baseline), 3))

share of frost days: 0.548 0.528 0.545
baseline on the validation stations | accuracy 0.528 | log loss 0.692


A little over half the days are frost days at each set of stations, 0.548 in training, 0.528 at the
validation stations and 0.545 at the test stations. Forecasting frost every day is right on the
validation days that had frost, an accuracy of 0.528, and its log loss is 0.692. Every candidate below
is measured against those two numbers.

### LightGBM's settings chosen on the validation stations

Three numbers of leaves, each trained with early stopping on the validation stations. A classifier
with two classes gets LightGBM's `binary_logloss` as its metric, which the last line prints. Each row
is scored on the validation stations, and `trees` is the number kept by early stopping.

In [3]:
def scored(probabilities, labels):
    return {"log loss": log_loss(labels, probabilities), "accuracy": accuracy_score(labels, probabilities > 0.5),
            "AUC": roc_auc_score(labels, probabilities)}

lightgbm_candidates, rows = {}, []
for leaves in (7, 31, 127):
    model = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, num_leaves=leaves, random_state=0, verbose=-1)
    model.fit(train[FEATURES], frost_train, eval_set=[(valid[FEATURES], frost_valid)],
              callbacks=[lgb.early_stopping(100, verbose=False)])
    lightgbm_candidates[leaves] = model
    rows.append({"num_leaves": leaves, "trees": model.best_iteration_,
                 **scored(model.predict_proba(valid[FEATURES])[:, 1], frost_valid)})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("metric:", list(lightgbm_candidates[31].best_score_["valid_0"]))

 num_leaves  trees  log loss  accuracy   AUC
          7    507     0.135     0.944 0.989
         31    178     0.131     0.947 0.990
        127    126     0.138     0.947 0.989
metric: ['binary_logloss']


All three candidates beat the baseline by a wide margin, with log losses from 0.131 to 0.138 against
its 0.692. Early stopping kept fewer trees as the trees grew larger, 507 with 7 leaves and 126 with
127. The three are close on every measure, with accuracies within 0.003 of each other and AUCs within
0.001. The 31-leaf model has the lowest validation log loss, 0.131, so it is the one kept. Its lead
over 7 leaves, 0.131 against 0.135, comes from one split of the stations, and another split could
reverse it.

### XGBoost's settings chosen the same way

Three depths, with early stopping on the validation stations' log loss, named with `eval_metric`.
XGBoost's `best_iteration` counts its rounds from 0, so the table adds one to it to give the number of
trees up to the best round. The last line checks that count against the round with the lowest
validation log loss in the model's own record.

In [4]:
xgboost_candidates, rows = {}, []
for depth in (3, 6, 9):
    model = xgb.XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=depth, early_stopping_rounds=100,
                              eval_metric="logloss", random_state=0)
    model.fit(train[FEATURES], frost_train, eval_set=[(valid[FEATURES], frost_valid)], verbose=False)
    xgboost_candidates[depth] = model
    rows.append({"max_depth": depth, "trees": model.best_iteration + 1,
                 **scored(model.predict_proba(valid[FEATURES])[:, 1], frost_valid)})
print(pd.DataFrame(rows).round(3).to_string(index=False))
record = xgboost_candidates[6].evals_result()["validation_0"]["logloss"]
print("depth 6, lowest log loss at round", int(np.argmin(record)), "counting from 0, of", len(record), "rounds trained")

 max_depth  trees  log loss  accuracy   AUC
         3    563     0.137     0.946 0.989
         6    260     0.130     0.948 0.990
         9    150     0.134     0.948 0.989
depth 6, lowest log loss at round 259 counting from 0, of 360 rounds trained


The XGBoost candidates fall in the same range, from 0.130 to 0.137, and depth 6 has the lowest, 0.130,
so it is the one kept, from one split of the stations as before. Its record holds 360 rounds, the 260
up to its best round and the 100 that early stopping waited through without a lower log loss. The
lowest is at round 259 counting from 0, which is why the table adds one. The next cell keeps the two
chosen models under names of their own.

In [5]:
lightgbm_frost = lightgbm_candidates[31]
xgboost_frost = xgboost_candidates[6]

### Probabilities against how often frost happened

If the model says 0.7 on a hundred days, frost should come on about seventy of them. The validation
days are grouped by the probability the chosen LightGBM model gave them, and each group's average
probability is set beside the share of its days that had frost.

In [6]:
probability = lightgbm_frost.predict_proba(valid[FEATURES])[:, 1]
band = pd.cut(probability, [0, 0.2, 0.4, 0.6, 0.8, 1.0], include_lowest=True)
checked = pd.DataFrame({"probability": probability, "frost": frost_valid.to_numpy()})
checked.groupby(band, observed=True).agg(days=("frost", "size"), forecast=("probability", "mean"),
                                          happened=("frost", "mean")).round(3)

,days,forecast,happened
"(-0.001, 0.2]",1112,0.027,0.031
"(0.2, 0.4]",67,0.283,0.433
"(0.4, 0.6]",55,0.488,0.582
"(0.6, 0.8]",85,0.718,0.718
"(0.8, 1.0]",1181,0.982,0.984


The two largest groups, which hold 2,293 of the 2,500 days, match closely. The days given less than
0.2 had an average forecast of 0.027 and frost on 0.031 of them, and the days given more than 0.8 had
0.982 against 0.984. In the groups from 0.2 to 0.4 and from 0.4 to 0.6, frost came more often than
forecast, 0.433 against 0.283 and 0.582 against 0.488. Those groups hold 67 and 55 days, and one split
cannot say how much of the gap would recur at other stations. Your turn runs the same check on XGBoost's model.

### The test stations measured once

The chosen models, and the baseline, on the 50 test stations.

In [7]:
test_rows = [
    {"forecast": "always frost", **scored(np.full(len(frost_test), frost_train.mean()), frost_test)},
    {"forecast": "LightGBM, 31 leaves", **scored(lightgbm_frost.predict_proba(test[FEATURES])[:, 1], frost_test)},
    {"forecast": "XGBoost, depth 6", **scored(xgboost_frost.predict_proba(test[FEATURES])[:, 1], frost_test)},
]
print(pd.DataFrame(test_rows).round(3).to_string(index=False))

           forecast  log loss  accuracy   AUC
       always frost     0.689     0.545 0.500
LightGBM, 31 leaves     0.150     0.933 0.987
   XGBoost, depth 6     0.151     0.935 0.987


On the test stations both models are far ahead of the baseline, with log losses of 0.150 and 0.151
against 0.689, and accuracies of 0.933 and 0.935 against 0.545. A forecast that gives every day the
same probability ranks no day above another, which is why the baseline's AUC is 0.5. The two libraries
are within 0.002 of each other on every measure, and one set of 50 stations gives no reason to prefer
either. Both log losses are higher than the chosen models' validation scores, 0.131 and 0.130. Part of
that gap may be the optimism described in The idea and part the difference between two sets of 50
stations, and one split cannot say how much is which. These are the numbers to report, and nothing above changes after
seeing them.

### What the forecast relies on

The average size of each feature's contribution to the chosen LightGBM model's raw score, over the
test days, in log-odds.

In [8]:
contributions = lightgbm_frost.predict(test[FEATURES], pred_contrib=True)[:, :-1]
reliance = pd.Series(np.abs(contributions).mean(axis=0), index=FEATURES)
print(reliance.sort_values(ascending=False, kind="stable").round(3).to_string())

day_of_year    3.560
snow_cm        1.314
latitude       1.257
elevation_m    0.380
cloud_cover    0.332
wind_ms        0.318
coast_km       0.144
terrain        0.124


The day of the year comes first, with an average contribution of 3.560, more than two and a half times
the next two, snow depth at 1.314 and latitude at 1.257. Elevation, cloud cover and wind follow, each
between 0.3 and 0.4, and distance from the coast and terrain move the forecast least. These are sizes
whatever their sign, so they say how far each feature moves the forecast, in either direction, and
they describe this model on these test days.

### Saving both models and forecasting a planned night

Both models are saved to `scratch`, loaded back, and checked against the models in memory on the test
stations.

In [9]:
lightgbm_frost.booster_.save_model(scratch / "frost_lightgbm.txt")
xgboost_frost.save_model(scratch / "frost_xgboost.json")
lightgbm_loaded = lgb.Booster(model_file=scratch / "frost_lightgbm.txt")
xgboost_loaded = xgb.XGBClassifier()
xgboost_loaded.load_model(scratch / "frost_xgboost.json")

print("LightGBM unchanged:", np.array_equal(lightgbm_loaded.predict(test[FEATURES]), lightgbm_frost.predict_proba(test[FEATURES])[:, 1]))
print("XGBoost unchanged:", np.array_equal(xgboost_loaded.predict_proba(test[FEATURES]), xgboost_frost.predict_proba(test[FEATURES])))

LightGBM unchanged: True
XGBoost unchanged: True


Both loaded models agree with the models in memory on all 2,500 test days. LightGBM's file comes back
as a `Booster`, whose `predict` gives the chance of frost, so it is compared with column 1 of the
classifier's `predict_proba`, and the two are identical. XGBoost's file is loaded into an
`XGBClassifier`, which keeps `predict_proba`, and its probabilities are identical as well.

The next cell asks both loaded models about a planned night at a valley station: latitude 62.0, 250
meters up, 60 kilometers from the coast, on April 30, day 120, with a clear sky, a wind of half a meter
a second and no snow on the ground.

In [10]:
night = pd.DataFrame({
    "latitude": [62.0], "elevation_m": [250], "coast_km": [60.0],
    "terrain": pd.Categorical(["valley"], categories=TERRAINS),
    "day_of_year": [120], "cloud_cover": [0.0], "wind_ms": [0.5], "snow_cm": [0.0],
})
print("LightGBM, chance of frost:", round(float(lightgbm_loaded.predict(night[FEATURES])[0]), 3))
print("XGBoost, chance of frost:", round(float(xgboost_loaded.predict_proba(night[FEATURES])[0, 1]), 3))

LightGBM, chance of frost: 0.856
XGBoost, chance of frost: 0.818


LightGBM gives the night a chance of frost of 0.856 and XGBoost 0.818. The two models were chosen and
trained separately, and both put the night above the threshold of one half used in the scores above,
so both forecast frost. Your turn asks the same models about a cloudy, windy version of this night.

## Your turn

Four tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/15-a-frost-forecast-solutions.ipynb).

**1.** Score the chosen LightGBM model on the validation stations at thresholds of 0.3, 0.5 and 0.7, and print the accuracy and the number of frost days it misses at each.

In [11]:
# your code here

**2.** Add a candidate with 15 leaves to LightGBM's comparison, trained the same way, and print its number of trees and its validation log loss beside the three in the worked examples.

In [12]:
# your code here

**3.** Group the validation days by the chosen XGBoost model's probabilities, in the same bands as the worked examples, and print each band's average probability and share of frost days.

In [13]:
# your code here

**4.** Ask both loaded models about the same planned night with a cloud cover of 1.0 and a wind of 8 meters a second, and print the two chances of frost beside the clear, calm night's.

In [14]:
# your code here

## Common errors

### No error, and a validation accuracy of 1.0

In [15]:
leaky_features = FEATURES + ["tmin"]
leaky = lgb.LGBMClassifier(n_estimators=100, random_state=0, verbose=-1).fit(train[leaky_features], frost_train)
print("validation accuracy:", accuracy_score(frost_valid, leaky.predict(valid[leaky_features])))

validation accuracy: 1.0


Every validation day is classified correctly, and the cell prints no warning. At LightGBM's default
verbosity the same fit prints
`[LightGBM] [Warning] No further splits with positive gain, best gain: -inf` once for each of its 100
trees, which `verbose=-1` hides, and that warning never names the feature that leaked. The label is whether
`tmin` is below 0, and `tmin` is one of the features, so the model can read every answer from it. On a real
evening the night's lowest temperature is what the forecast is for, so the column would not exist yet.
The fix is to check that no feature is the target or is made from it, and to train on `FEATURES`
alone, as the next cell does.

In [16]:
honest = lgb.LGBMClassifier(n_estimators=100, random_state=0, verbose=-1).fit(train[FEATURES], frost_train)
print("validation accuracy without tmin:", round(accuracy_score(frost_valid, honest.predict(valid[FEATURES])), 3))

validation accuracy without tmin: 0.948


Without `tmin`, the same small model is right on 0.948 of the validation days, close to the chosen
models' 0.947 and 0.948.

### ``ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1], got ['frost' 'thaw']``

In [17]:
as_words = np.where(train["tmin"] < 0, "frost", "thaw")
xgb.XGBClassifier(n_estimators=10).fit(train[FEATURES], as_words)

ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1], got ['frost' 'thaw']

XGBoost's classifier takes as labels only the numbers from 0 to one less than the number of classes,
and here the labels arrived as the words `frost` and `thaw`. The fix is to turn the words into those
numbers before fitting, with frost as 1, so that column 1 of `predict_proba` is the chance of frost.
The next cell runs the fix, and then gives the same words to LightGBM.

In [18]:
frost_as_number = (as_words == "frost").astype(int)
words_xgboost = xgb.XGBClassifier(n_estimators=10, random_state=0).fit(train[FEATURES], frost_as_number)
print("XGBoost's classes:", words_xgboost.classes_)

words_lightgbm = lgb.LGBMClassifier(n_estimators=10, random_state=0, verbose=-1).fit(train[FEATURES], as_words)
print("LightGBM's classes:", words_lightgbm.classes_)
print("LightGBM's column 0, averaged over the training days:", round(float(words_lightgbm.predict_proba(train[FEATURES])[:, 0].mean()), 3))

XGBoost's classes: [0 1]
LightGBM's classes: ['frost' 'thaw']
LightGBM's column 0, averaged over the training days: 0.547


XGBoost fits once the labels are numbers. LightGBM accepted the words and gave its classes in
alphabetical order, `frost` then `thaw`, so in its `predict_proba` the chance of frost is column 0.
That column averages 0.547 over the training days, close to the share of frost days in training,
0.548. A project that trains one library on words and the other on numbers ends up with frost in different
columns, which is the mistake under the next heading.

### No error, and the probability of the other class

In [19]:
small_frost = xgb.XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, random_state=0)
small_frost.fit(train[FEATURES], (train["tmin"] < 0).astype(int))
spring_night = pd.DataFrame({
    "latitude": [62.0], "elevation_m": [250], "coast_km": [60.0],
    "terrain": pd.Categorical(["valley"], categories=TERRAINS),
    "day_of_year": [120], "cloud_cover": [0.0], "wind_ms": [0.5], "snow_cm": [0.0],
})
print("chance of frost, read from column 0:", round(float(small_frost.predict_proba(spring_night)[0, 0]), 3))

chance of frost, read from column 0: 0.179


The cell trains a small model of its own and asks it about the spring night from the worked examples.
Read from column 0, the night's chance of frost looks like 0.179, low enough to leave the roads
ungritted. `predict_proba` returns one column for each class, in the order of `classes_`, and column 0
belongs to class 0, a thaw. The fix is to find the column from `classes_`.

In [20]:
print("classes, in column order:", small_frost.classes_)
frost_column = list(small_frost.classes_).index(1)
print(f"chance of frost, read from column {frost_column}:", round(float(small_frost.predict_proba(spring_night)[0, frost_column]), 3))

classes, in column order: [0 1]
chance of frost, read from column 1: 0.821


Read from column 1, the chance of frost is 0.821, and the two columns add to one. Looking up frost's
label in `classes_` finds its column wherever the labels put it. Here the label is 1, and for the
LightGBM model trained on words it is `"frost"`, in column 0.

In [21]:
shutil.rmtree(scratch)
print("scratch removed:", not scratch.exists())

scratch removed: True


## Recap

- You can build a frost forecast in an order that gives the training, validation and test stations
  one job each.
- You can choose each library's settings on the validation stations by log loss, and measure the
  chosen model once on the test stations.
- You can check a model's probabilities against how often frost happened, and read which features it
  relies on.
- You can save both models, load them back, check them, and forecast a night they have never seen.

## What is next

The **GeoPandas, Deep Dive** guide puts geometry into a pandas table, for spatial joins and coordinate
systems, and the library's front page lists every guide.

---

&#8592; **Previous:** [Saving and Loading Boosters](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/14-saving-and-loading-boosters.ipynb)  &nbsp;·&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)
